# Tweet sentiment — TF-IDF exploration + PyTorch LSTM (GPU)


In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path('tweet_sentiment.csv')
if not csv_path.exists():
    csv_path = Path('/content/tweet_sentiment.csv')
df = pd.read_csv(csv_path)
df.head()


In [ ]:
df.isna().sum()


In [ ]:
df['cleaned_tweets'] = df['cleaned_tweets'].fillna(df['cleaned_tweets'].mode()[0])


In [ ]:
df.isna().sum()


In [ ]:
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import word_tokenize


In [ ]:
stemmer = PorterStemmer()


In [ ]:
import nltk
nltk.download('punkt_tab')


In [ ]:
df['tokens'] = df['cleaned_tweets'].apply(word_tokenize)


In [ ]:
df['stemmed_tokens'] = df['tokens'].apply(lambda x: [stemmer.stem(word) for word in x])


In [ ]:
df['stemmed_text'] = df['stemmed_tokens'].apply(lambda x: ' '.join(x))


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
x_tfidf = tfidf.fit_transform(df['stemmed_text'])
print(x_tfidf.shape)


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(df['sentiment'])
print(le.classes_)


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

class SimpleTokenizer:
    """Word tokenizer (word_index starts at 1; 0 is padding)."""

    def __init__(self):
        self.word_index = {}

    def fit_on_texts(self, texts):
        vocab = {}
        for text in texts:
            for word in str(text).lower().split():
                vocab[word] = vocab.get(word, 0) + 1
        # Sort by frequency (desc), then word (asc) — freq-sorted vocab
        sorted_words = sorted(vocab.items(), key=lambda x: (-x[1], x[0]))
        self.word_index = {w: i + 1 for i, (w, _) in enumerate(sorted_words)}
        return self

    def texts_to_sequences(self, texts):
        sequences = []
        for text in texts:
            seq = [self.word_index[w] for w in str(text).lower().split() if w in self.word_index]
            sequences.append(seq)
        return sequences


def pad_sequences(sequences, maxlen=100, value=0):
    out = np.full((len(sequences), maxlen), value, dtype=np.int64)
    for i, seq in enumerate(sequences):
        if not seq:
            continue
        trunc = seq[-maxlen:]
        out[i, -len(trunc):] = trunc
    return out


In [ ]:
tokenizer = SimpleTokenizer()
tokenizer.fit_on_texts(df['cleaned_tweets'])


In [ ]:
x = tokenizer.texts_to_sequences(df['cleaned_tweets'])
x = pad_sequences(x, maxlen=100)


In [ ]:
df['sentiment'] = df['sentiment'].replace(-1, 2)
y = df['sentiment'].astype(int).values
print(df['sentiment'].unique())


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
vocab_size = len(tokenizer.word_index) + 1
print('vocab_size:', vocab_size)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=64, num_classes=3, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x: (batch, seq_len)
        emb = self.embedding(x)
        emb = self.dropout(emb)
        out, (h_n, _) = self.lstm(emb)
        h = self.dropout(h_n[-1])  # last layer hidden state
        return self.fc(h)


model = SentimentLSTM(vocab_size=vocab_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

x_train_t = torch.tensor(np.asarray(x_train), dtype=torch.long)
y_train_t = torch.tensor(np.asarray(y_train), dtype=torch.long)
x_test_t = torch.tensor(np.asarray(x_test), dtype=torch.long)
y_test_t = torch.tensor(np.asarray(y_test), dtype=torch.long)

train_loader = DataLoader(TensorDataset(x_train_t, y_train_t), batch_size=16, shuffle=True)
val_loader = DataLoader(TensorDataset(x_test_t, y_test_t), batch_size=16)

EPOCHS = 10
PATIENCE = 3
best_val_loss = float("inf")
patience_left = PATIENCE
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)
        train_correct += (logits.argmax(1) == yb).sum().item()
        train_total += xb.size(0)

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            val_loss += loss.item() * xb.size(0)
            val_correct += (logits.argmax(1) == yb).sum().item()
            val_total += xb.size(0)

    train_loss /= max(train_total, 1)
    val_loss /= max(val_total, 1)
    train_acc = train_correct / max(train_total, 1)
    val_acc = val_correct / max(val_total, 1)
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} acc={val_acc:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_left = PATIENCE
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_left -= 1
        if patience_left == 0:
            print("Early stopping.")
            break

if best_state is not None:
    model.load_state_dict(best_state)
    model.to(device)
    print("Restored best validation weights.")


In [ ]:
model.eval()
with torch.no_grad():
    logits = model(x_test_t.to(device))
    preds = logits.argmax(1).cpu().numpy()
    acc = (preds == y_test).mean()
print('Test accuracy:', acc)
